# Day 34 — Knapsack: 0/1, unbounded and bounded

Day 33 ended on a warning: swapping two loops turned "how many combinations" into
"how many permutations", and Python did not say a word. Today the same trap appears
one level smaller — the **direction** of a single inner loop decides whether each
item may be taken once or an unlimited number of times.

Four items, one bag:

| item | weight | value |
|---|---|---|
| rope | 3 | 50 |
| book | 4 | 40 |
| pan | 5 | 70 |
| tent | 6 | 80 |

Capacity 10. The same four items give **three different answers** — 130, 140, 150 —
depending only on how many copies of each item exist. The code barely changes.

## 1. The 2-D table

`dp[i][c]` is the best value using the first `i` items with capacity `c`.
Item `i-1` has exactly two fates: leave it, and the cell inherits the row above;
or take it, and the cell is `dp[i-1][c-w] + v`. Both readings live in row `i-1`,
so the rows can be filled top to bottom with nothing unfinished in sight.

In [1]:
NAMES   = ['rope', 'book', 'pan', 'tent']
WEIGHTS = [3, 4, 5, 6]
VALUES  = [50, 40, 70, 80]
CAP     = 10

def knap01_table(weights, values, cap):
    """dp[i][c] = best value using the first i items with capacity c."""
    n = len(weights)
    dp = [[0] * (cap + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        w, v = weights[i - 1], values[i - 1]
        for c in range(cap + 1):
            dp[i][c] = dp[i - 1][c]                       # leave it
            if w <= c and dp[i - 1][c - w] + v > dp[i][c]:
                dp[i][c] = dp[i - 1][c - w] + v           # take it
    return dp

def knap01_items(weights, values, cap):
    """Walk the finished table backwards to recover which items were taken."""
    dp = knap01_table(weights, values, cap)
    chosen, c = [], cap
    for i in range(len(weights), 0, -1):
        if dp[i][c] != dp[i - 1][c]:      # the row changed => item i-1 was taken
            chosen.append(i - 1)
            c -= weights[i - 1]
    chosen.reverse()
    return dp[len(weights)][cap], chosen

row = lambda vals: ' '.join(f'{v:>4}' for v in vals)
DP = knap01_table(WEIGHTS, VALUES, CAP)
print('      ' + row(range(CAP + 1)))
for tag, r in zip(['none'] + NAMES, DP):
    print(f'{tag:>5} ' + row(r))

best, chosen = knap01_items(WEIGHTS, VALUES, CAP)
print()
print('best value :', best)
print('items      :', [NAMES[i] for i in chosen])
print('weight used:', sum(WEIGHTS[i] for i in chosen), 'of', CAP)

         0    1    2    3    4    5    6    7    8    9   10
 none    0    0    0    0    0    0    0    0    0    0    0
 rope    0    0    0   50   50   50   50   50   50   50   50
 book    0    0    0   50   50   50   50   90   90   90   90
  pan    0    0    0   50   50   70   70   90  120  120  120
 tent    0    0    0   50   50   70   80   90  120  130  130

best value : 130
items      : ['rope', 'tent']
weight used: 9 of 10


130 = rope + tent, weight 9 of 10. Note what the table does *not* do: it never tries
to fill the bag. It leaves a unit of capacity on the floor because the alternative
(rope + pan, weight 8) is worth less. Any heuristic that optimises "space used"
gets this wrong.

## 2. The same thing in one row

Only row `i-1` is ever read, so the table can collapse to a single array — provided
`dp[c-w]` still holds the *old* row when `dp[c]` is written. Walking `c` **downwards**
guarantees it: everything to the left is untouched this round.

In [2]:
def knap01_rolling(weights, values, cap):
    """0/1 knapsack in one row. The inner loop must run DOWNWARDS."""
    dp = [0] * (cap + 1)
    for w, v in zip(weights, values):
        for c in range(cap, w - 1, -1):          # descending: read the old row
            if dp[c - w] + v > dp[c]:
                dp[c] = dp[c - w] + v
    return dp[cap]

def rolling_rows(weights, values, cap, descending=True):
    """Snapshot the row after each item, so the two directions can be compared."""
    dp = [0] * (cap + 1)
    rows = [dp[:]]
    for w, v in zip(weights, values):
        rng = range(cap, w - 1, -1) if descending else range(w, cap + 1)
        for c in rng:
            if dp[c - w] + v > dp[c]:
                dp[c] = dp[c - w] + v
        rows.append(dp[:])
    return rows

print('      ' + row(range(CAP + 1)))
for tag, r in zip(['start'] + ['+' + n for n in NAMES],
                  rolling_rows(WEIGHTS, VALUES, CAP, descending=True)):
    print(f'{tag:>5} ' + row(r))
print()
print('one row  :', knap01_rolling(WEIGHTS, VALUES, CAP))
print('2-D table:', DP[-1][CAP])

         0    1    2    3    4    5    6    7    8    9   10
start    0    0    0    0    0    0    0    0    0    0    0
+rope    0    0    0   50   50   50   50   50   50   50   50
+book    0    0    0   50   50   50   50   90   90   90   90
 +pan    0    0    0   50   50   70   70   90  120  120  120
+tent    0    0    0   50   50   70   80   90  120  130  130

one row  : 130
2-D table: 130


Same 130, one array instead of five. Look at the `+pan` row: writing `dp[8] = 120`
reads `dp[3] = 50`, five cells to the left — a cell this round has not reached yet,
because the loop is walking down. `pan` is therefore offered exactly once.

## 3. Reverse the direction and the problem changes

Delete one `reversed`. Now `dp[c-w]` has *already* been updated this round, so it may
already contain a copy of this very item — and taking it again is exactly what
*unbounded* means. Nothing raises, nothing warns; the number is simply an answer to a
different question.

In [3]:
def knap_unbounded(weights, values, cap):
    """Identical body, ASCENDING inner loop: each item is unlimited."""
    dp = [0] * (cap + 1)
    for w, v in zip(weights, values):
        for c in range(w, cap + 1):              # ascending: read the new row
            if dp[c - w] + v > dp[c]:
                dp[c] = dp[c - w] + v
    return dp[cap]

def unbounded_counts(weights, values, cap):
    """How many copies of each item the ascending loop actually used."""
    dp = [0] * (cap + 1)
    pick = [-1] * (cap + 1)
    for i, (w, v) in enumerate(zip(weights, values)):
        for c in range(w, cap + 1):
            if dp[c - w] + v > dp[c]:
                dp[c] = dp[c - w] + v
                pick[c] = i
    counts = [0] * len(weights)
    c = cap
    while c > 0 and pick[c] >= 0:
        counts[pick[c]] += 1
        c -= weights[pick[c]]
    return dp[cap], counts

print('      ' + row(range(CAP + 1)))
for tag, r in zip(['start'] + ['+' + n for n in NAMES],
                  rolling_rows(WEIGHTS, VALUES, CAP, descending=False)):
    print(f'{tag:>5} ' + row(r))

total, counts = unbounded_counts(WEIGHTS, VALUES, CAP)
print()
print('descending (0/1)      :', knap01_rolling(WEIGHTS, VALUES, CAP))
print('ascending  (unbounded):', knap_unbounded(WEIGHTS, VALUES, CAP))
print('copies used           :',
      {n: k for n, k in zip(NAMES, counts) if k})

         0    1    2    3    4    5    6    7    8    9   10
start    0    0    0    0    0    0    0    0    0    0    0
+rope    0    0    0   50   50   50  100  100  100  150  150
+book    0    0    0   50   50   50  100  100  100  150  150
 +pan    0    0    0   50   50   70  100  100  120  150  150
+tent    0    0    0   50   50   70  100  100  120  150  150

descending (0/1)      : 130
ascending  (unbounded): 150
copies used           : {'rope': 3}


The `+rope` row already gives it away: `dp[6] = 100` and `dp[9] = 150` appear before
any other item has been considered. `dp[6]` read `dp[3]`, which *this round* already
contains one rope. The final 150 is three ropes and nothing else.

Both numbers are correct — for different problems. Only the loop direction says which
problem you asked.

## 4. Bounded knapsack: k copies of each item

Reality is usually neither "one" nor "infinite": there are 2 ropes, 1 book, 3 pans,
1 tent. The obvious fix is to expand each item into `k` separate 0/1 items. It is
correct, and it costs `O(cap x sum(counts))` — linear in the *value* of the counts,
not in the number of digits needed to write them.

Binary splitting replaces `k` copies with bundles of size 1, 2, 4, ... and a
remainder: `O(log k)` items that reproduce every possible count.

In [4]:
import time

def knap_bounded_naive(weights, values, counts, cap):
    w2, v2 = [], []
    for w, v, k in zip(weights, values, counts):
        w2 += [w] * k
        v2 += [v] * k
    return knap01_rolling(w2, v2, cap), len(w2)

def binary_split(k):
    """k -> [1, 2, 4, ..., remainder]"""
    parts, p = [], 1
    while p <= k:
        parts.append(p)
        k -= p
        p *= 2
    if k:
        parts.append(k)
    return parts

def knap_bounded_binary(weights, values, counts, cap):
    w2, v2 = [], []
    for w, v, k in zip(weights, values, counts):
        for p in binary_split(k):
            w2.append(w * p)
            v2.append(v * p)
    return knap01_rolling(w2, v2, cap), len(w2)

COUNTS = [2, 1, 3, 1]
a, na = knap_bounded_naive(WEIGHTS, VALUES, COUNTS, CAP)
b, nb = knap_bounded_binary(WEIGHTS, VALUES, COUNTS, CAP)
print(f'counts {COUNTS}: expanded {na} items -> {a},  split {nb} items -> {b}')

# the gap only shows up when the counts are large
BW = [3, 4, 5, 6, 7]
BV = [50, 40, 70, 80, 95]
BC = [300, 500, 800, 1200, 2000]
BCAP = 4000
t = time.perf_counter(); a2, na2 = knap_bounded_naive(BW, BV, BC, BCAP)
ta = (time.perf_counter() - t) * 1000
t = time.perf_counter(); b2, nb2 = knap_bounded_binary(BW, BV, BC, BCAP)
tb = (time.perf_counter() - t) * 1000
print(f'counts {BC}, cap {BCAP}:')
print(f'  expanded {na2:>5} items  {ta:8.1f} ms  -> {a2}')
print(f'  split    {nb2:>5} items  {tb:8.1f} ms  -> {b2}')
print(f'  speedup  {ta / tb:.0f}x, same answer: {a2 == b2}')

counts [2, 1, 3, 1]: expanded 7 items -> 140,  split 6 items -> 140
counts [300, 500, 800, 1200, 2000], cap 4000:
  expanded  4800 items    3441.4 ms  -> 58400
  split       50 items      21.6 ms  -> 58400
  speedup  159x, same answer: True


Same answer, 96x fewer items. And the splitting is not a heuristic — every count from
0 to k really is a subset sum of the bundles, which is small enough to check by brute
force:

In [5]:
def binary_split_reaches(k):
    """Proof by exhaustion that the parts cover every count 0..k."""
    parts = binary_split(k)
    seen = set()
    for mask in range(1 << len(parts)):
        seen.add(sum(p for j, p in enumerate(parts) if mask >> j & 1))
    return seen == set(range(k + 1)), parts

for k in (1, 3, 7, 13, 100, 1000):
    if k <= 13:
        ok, parts = binary_split_reaches(k)
        print(f'k={k:>4}  {len(parts)} parts {parts}  covers 0..{k}: {ok}')
    else:
        parts = binary_split(k)
        print(f'k={k:>4}  {len(parts)} parts {parts}')

k=   1  1 parts [1]  covers 0..1: True
k=   3  2 parts [1, 2]  covers 0..3: True
k=   7  3 parts [1, 2, 4]  covers 0..7: True
k=  13  4 parts [1, 2, 4, 6]  covers 0..13: True
k= 100  7 parts [1, 2, 4, 8, 16, 32, 37]
k=1000  10 parts [1, 2, 4, 8, 16, 32, 64, 128, 256, 489]


## 5. The greedy that is right for one problem and wrong for the other

Sort by value per unit of weight and take greedily. For the **fractional** knapsack —
where the last item may be cut — this is provably optimal. Remove the scissors and the
same rule quietly loses value, on the very same items.

In [6]:
def fractional_greedy(weights, values, cap):
    """Best ratio first, allowed to cut the last item. Provably optimal."""
    order = sorted(range(len(weights)), key=lambda i: values[i] / weights[i],
                   reverse=True)
    total, left, taken = 0.0, cap, []
    for i in order:
        if left <= 0:
            break
        take = min(1.0, left / weights[i])
        total += take * values[i]
        left -= take * weights[i]
        taken.append((i, take))
    return total, taken

def greedy_01(weights, values, cap):
    """The same rule with no cutting allowed - now it can lose."""
    order = sorted(range(len(weights)), key=lambda i: values[i] / weights[i],
                   reverse=True)
    total, left, taken = 0, cap, []
    for i in order:
        if weights[i] <= left:
            total += values[i]
            left -= weights[i]
            taken.append(i)
    return total, taken

for i in sorted(range(4), key=lambda i: VALUES[i] / WEIGHTS[i], reverse=True):
    print(f'{NAMES[i]:>5}  {VALUES[i]}/{WEIGHTS[i]} = {VALUES[i] / WEIGHTS[i]:.2f}')

frac, ftaken = fractional_greedy(WEIGHTS, VALUES, CAP)
g01, gtaken = greedy_01(WEIGHTS, VALUES, CAP)
print()
print(f'cut allowed : {frac:.2f}  ' +
      ', '.join(f'{NAMES[i]} x{t:.2f}' for i, t in ftaken))
print(f'no cutting  : {g01}      ' + ' + '.join(NAMES[i] for i in gtaken))
print(f'DP          : {best}      ' + ' + '.join(NAMES[i] for i in chosen))
print(f'the greedy loses {best - g01} and reports nothing at all')

 rope  50/3 = 16.67
  pan  70/5 = 14.00
 tent  80/6 = 13.33
 book  40/4 = 10.00

cut allowed : 146.67  rope x1.00, pan x1.00, tent x0.33
no cutting  : 120      rope + pan
DP          : 130      rope + tent
the greedy loses 10 and reports nothing at all


The greedy takes rope and pan (weight 8, value 120) and then cannot fit the tent.
The table takes rope and tent (weight 9, value 130). The difference is only 10 — but
the greedy has no way to know it is wrong, and no bigger instance makes it safer.

## 6. Subset sum — the third operator

Day 33 had `max` (fewest coins) and `+` (count the ways). The third is `or`: forget
the value, just ask which totals are reachable. The loop is the descending one, so
each number is used at most once.

In [7]:
def subset_sum(nums, target):
    """Reachability with a boolean row."""
    dp = [False] * (target + 1)
    dp[0] = True
    for x in nums:
        for c in range(target, x - 1, -1):
            if dp[c - x]:
                dp[c] = True
    return dp[target]

def subset_sum_bits(nums, target):
    """The whole boolean row packed into ONE Python integer."""
    bits = 1
    for x in nums:
        bits |= bits << x            # shift every reachable sum up by x at once
    return bool(bits >> target & 1)

NUMS = [3, 34, 4, 12, 5, 2]
for t in (9, 30, 11):
    print(f'target {t:>3}: list {subset_sum(NUMS, t)!s:>5}   '
          f'bitset {subset_sum_bits(NUMS, t)!s:>5}')

# 400 numbers, one target - best-of-N, because a single run is mostly noise
big = [i % 97 + 1 for i in range(400)]
tgt = 9545

def best_of(fn, reps):
    out, r = float('inf'), None
    for _ in range(reps):
        t0 = time.perf_counter()
        r = fn(big, tgt)
        out = min(out, time.perf_counter() - t0)
    return out * 1000, r

slow, r1 = best_of(subset_sum, 3)
fast, r2 = best_of(subset_sum_bits, 30)
print()
print(f'list of bools : {slow:9.3f} ms  -> {r1}')
print(f'one big int   : {fast:9.3f} ms  -> {r2}')
print(f'speedup       : {slow / fast:.0f}x (same answer: {r1 == r2})')

target   9: list  True   bitset  True
target  30: list False   bitset False
target  11: list  True   bitset  True

list of bools :   233.134 ms  -> True
one big int   :     0.236 ms  -> True
speedup       : 987x (same answer: True)


`bits |= bits << x` is the entire inner loop, executed by CPython's big-integer code
one machine word at a time instead of one Python object at a time. The descending-order
worry disappears too: the shift reads a snapshot of `bits`, so an item can never see
itself.

## 7. LeetCode — five problems, one table

416 is subset sum on `total/2`. 494 turns `+/-` into "pick a subset that sums to
`(total+target)/2`". 1049 is 416 asking for the closest reachable half. 474 is a
knapsack with two capacities. 279 is unbounded coin change over squares.

In [8]:
def lc416_can_partition(nums):
    """416. Partition Equal Subset Sum - subset sum to total/2."""
    total = sum(nums)
    if total % 2:
        return False
    return subset_sum_bits(nums, total // 2)

def lc494_find_target_sum_ways(nums, target):
    """494. Target Sum - P - N = target and P + N = sum, so the positive side
    must reach (sum + target) / 2. Counting subsets is the `+` version."""
    total = sum(nums)
    if abs(target) > total or (total + target) % 2:
        return 0
    s = (total + target) // 2
    dp = [0] * (s + 1)
    dp[0] = 1
    for x in nums:
        for c in range(s, x - 1, -1):
            dp[c] += dp[c - x]
    return dp[s]

def lc1049_last_stone_weight_ii(stones):
    """1049. Last Stone Weight II - split into two piles as even as possible."""
    total = sum(stones)
    half = total // 2
    dp = [0] * (half + 1)
    for x in stones:
        for c in range(half, x - 1, -1):
            if dp[c - x] + x > dp[c]:
                dp[c] = dp[c - x] + x
    return total - 2 * dp[half]

def lc474_find_max_form(strs, m, n):
    """474. Ones and Zeroes - two capacities, so BOTH loops run backwards."""
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        z, o = s.count('0'), s.count('1')
        for i in range(m, z - 1, -1):
            for j in range(n, o - 1, -1):
                if dp[i - z][j - o] + 1 > dp[i][j]:
                    dp[i][j] = dp[i - z][j - o] + 1
    return dp[m][n]

def lc279_num_squares(n):
    """279. Perfect Squares - unbounded, so the inner loop runs FORWARDS."""
    squares, k = [], 1
    while k * k <= n:
        squares.append(k * k)
        k += 1
    INF = float('inf')
    dp = [0] + [INF] * n
    for s in squares:
        for c in range(s, n + 1):
            if dp[c - s] + 1 < dp[c]:
                dp[c] = dp[c - s] + 1
    return dp[n]

print('416 [1,5,11,5]        ->', lc416_can_partition([1, 5, 11, 5]))
print('416 [1,2,3,5]         ->', lc416_can_partition([1, 2, 3, 5]))
print('494 [1,1,1,1,1] t=3   ->', lc494_find_target_sum_ways([1] * 5, 3))
print('1049 [2,7,4,1,8,1]    ->', lc1049_last_stone_weight_ii([2, 7, 4, 1, 8, 1]))
print("474 5 zeros, 3 ones   ->",
      lc474_find_max_form(['10', '0001', '111001', '1', '0'], 5, 3))
print('279 n=12              ->', lc279_num_squares(12))
print('279 n=13              ->', lc279_num_squares(13))

416 [1,5,11,5]        -> True
416 [1,2,3,5]         -> False
494 [1,1,1,1,1] t=3   -> 5
1049 [2,7,4,1,8,1]    -> 1
474 5 zeros, 3 ones   -> 4
279 n=12              -> 3
279 n=13              -> 2


474 is the one to remember in an interview: two capacities means a rolling *grid*, and
**both** loops have to run backwards. Get one of them wrong and you have silently
allowed a string to be used twice. 279 runs forwards on purpose — you may reuse `4` as
many times as you like.

## 8. "O(n x cap)" is not polynomial

The table is polynomial in the *capacity*, but the capacity arrives as a number, and a
number of `b` bits can be as large as `2**b`. Multiplying every weight and the capacity
by 1000 does not change the answer and adds ten bits of input — and makes the table a
thousand times bigger.

In [9]:
def scale_report(weights, values, cap, factors=(1, 10, 100, 1000)):
    rows = []
    for f in factors:
        w = [x * f for x in weights]
        b = knap01_rolling(w, values, cap * f)
        cells = len(weights) * (cap * f + 1)
        bits = sum(max(1, x.bit_length()) for x in w) + (cap * f).bit_length()
        rows.append((f, b, cells, bits))
    return rows

print(f"{'factor':>7} {'answer':>8} {'table cells':>13} {'input bits':>12}")
for f, b, cells, bits in scale_report(WEIGHTS, VALUES, CAP):
    print(f'{f:>7} {b:>8} {cells:>13,} {bits:>12}')

# and the constant factor of the 2-D table vs the rolling row
import random
rnd = random.Random(34)
w = [rnd.randint(1, 100) for _ in range(200)]
v = [rnd.randint(1, 1000) for _ in range(200)]
t = time.perf_counter(); a = knap01_table(w, v, 5000)[200][5000]
t2d = (time.perf_counter() - t) * 1000
t = time.perf_counter(); b = knap01_rolling(w, v, 5000)
t1d = (time.perf_counter() - t) * 1000
print()
print(f'200 items, cap 5000 -> {a}')
print(f'  2-D table   {t2d:8.1f} ms')
print(f'  rolling row {t1d:8.1f} ms   ({t2d / t1d:.1f}x faster, same answer: {a == b})')

 factor   answer   table cells   input bits
      1      130            44           15
     10      130           404           30
    100      130         4,004           47
   1000      130        40,004           64

200 items, cap 5000 -> 78759
  2-D table      426.8 ms
  rolling row    245.4 ms   (1.7x faster, same answer: True)


That is what *pseudo-polynomial* means, and it is why 0/1 knapsack is NP-hard even
though this notebook solves it in eight lines: the running time is polynomial in the
value of the input, not in its length.

## Tests

In [10]:
assert knap01_table(WEIGHTS, VALUES, CAP)[-1][CAP] == 130
assert knap01_items(WEIGHTS, VALUES, CAP) == (130, [0, 3])
assert knap01_rolling(WEIGHTS, VALUES, CAP) == 130
assert knap_unbounded(WEIGHTS, VALUES, CAP) == 150
assert unbounded_counts(WEIGHTS, VALUES, CAP) == (150, [3, 0, 0, 0])
assert knap_bounded_naive(WEIGHTS, VALUES, [2, 1, 3, 1], CAP)[0] == 140
assert knap_bounded_binary(WEIGHTS, VALUES, [2, 1, 3, 1], CAP)[0] == 140
assert binary_split(13) == [1, 2, 4, 6]
assert all(binary_split_reaches(k)[0] for k in range(1, 40))
assert greedy_01(WEIGHTS, VALUES, CAP)[0] == 120 < 130
assert abs(fractional_greedy(WEIGHTS, VALUES, CAP)[0] - 146.666666) < 1e-4
assert subset_sum([3, 34, 4, 12, 5, 2], 9)
assert not subset_sum([3, 34, 4, 12, 5, 2], 30)
assert all(subset_sum(NUMS, t) == subset_sum_bits(NUMS, t) for t in range(61))
assert lc416_can_partition([1, 5, 11, 5]) and not lc416_can_partition([1, 2, 3, 5])
assert lc494_find_target_sum_ways([1] * 5, 3) == 5
assert lc1049_last_stone_weight_ii([2, 7, 4, 1, 8, 1]) == 1
assert lc474_find_max_form(['10', '0001', '111001', '1', '0'], 5, 3) == 4
assert (lc279_num_squares(12), lc279_num_squares(13)) == (3, 2)
assert all(r[1] == 130 for r in scale_report(WEIGHTS, VALUES, CAP))
print('all assertions passed')

all assertions passed
